In [1]:
# pip install -qU langchain "langchain[google-genai]"
from langchain.agents import create_agent

In [4]:
import os

os.environ["GOOGLE_API_KEY"] = ""

In [10]:

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather in Chennai?"
            }
        ]
    }
)

content = result["messages"][-1].content

if isinstance(content, list):
    text = "".join(
        block.get("text", "")
        for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    )
else:
    text = content

print(text)

The weather in Chennai is always sunny!


In [11]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [12]:
import requests
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current real weather for a city."""

    # Convert city name to coordinates
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    geo_response = requests.get(geo_url, params=geo_params)
    geo_data = geo_response.json()

    if "results" not in geo_data:
        return f"Could not find the city: {city}"

    location = geo_data["results"][0]
    latitude = location["latitude"]
    longitude = location["longitude"]

    # Get actual weather
    weather_url = "https://api.open-meteo.com/v1/forecast"
    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,apparent_temperature,weather_code,wind_speed_10m",
        "timezone": "auto"
    }

    weather_response = requests.get(weather_url, params=weather_params)
    weather_data = weather_response.json()

    current = weather_data["current"]

    return (
        f"Weather in {location['name']}, {location.get('country', '')}:\n"
        f"Temperature: {current['temperature_2m']} °C\n"
        f"Feels like: {current['apparent_temperature']} °C\n"
        f"Humidity: {current['relative_humidity_2m']}%\n"
        f"Wind: {current['wind_speed_10m']} km/h\n"
        f"Weather code: {current['weather_code']}"
    )

In [13]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
)

In [14]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What's the current weather in Schwäbisch Hall?"
        }
    ]
})

content = result["messages"][-1].content

if isinstance(content, list):
    text = "".join(
        block.get("text", "")
        for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    )
else:
    text = content

print(text)

The current weather in Schwäbisch Hall, Germany is 18.0 °C (feels like 17.6 °C) with 66% humidity and a wind speed of 5.8 km/h.
